# OCM Catalyst — Using Literature Data Without Damaging Accuracy

## Chapter 1: The Problem

**OCM (Oxidative Coupling of Methane)** is a catalytic reaction that converts CH₄ + O₂ into ethylene (C₂). The goal is to predict the C₂ yield (Y, in %) from catalyst composition and reaction temperature.

---

### Two datasets

| | Our lab (2025) | Published literature |
|---|---|---|
| **Size** | 89,074 rows | 3,852 rows |
| **Preparation method** | Impregnation only | 15+ methods |
| **Mean Y(C2)** | 5.25% | 8.67% |
| **Chemistry** | Consistent, one lab | 40 years of diverse labs |

**The central question:** Can we use published literature to improve our model's accuracy on our own experiments — without hurting it?

---

### Three reasons it is non-trivial

1. **Label shift** — Literature mean yield (8.67%) is 3.42 percentage points higher than ours (5.25%). This is a *systematic* offset caused by publication bias (only successful experiments get published) and different operating conditions (gas flow rates, CH₄:O₂ ratios are optimised in literature but fixed in our lab).

2. **Covariate shift** — 78.5% of literature samples describe catalyst compositions that are chemically unlike anything in our dataset. Adding them trains the model on chemistry it will never see in our lab.

3. **Publication bias** — Papers preferentially report conditions that maximise yield. The distribution of literature labels does not represent a random draw from the chemistry space.

The three figures in Chapter 2 visualise problems #1 (label shift), #2 (covariate shift), and the concrete chemical asymmetry between the two datasets.

---

### The narrative thread

We will try 5 approaches. Each one solves part of the problem but reveals a new one:

| Method | Core idea | What it leaves unsolved |
|---|---|---|
| Baseline | Our data only | No transfer learning |
| Naive merge | Append all literature | Label shift corrupts the model |
| DRST | Filter by chemical similarity (hard threshold) | Hard binary cutoff is arbitrary |
| KMM | Soft weights per literature sample | Systematic offset still bleeds in |
| **Prior Feature Transfer** | Literature as a prior feature, not a label | — |

---

> **Intuition:** Think of it like this: we have our own recipe book with 89,000 experiments, all done the same way in our kitchen. There is also a public recipe book from 40 years of different labs worldwide — 3,852 recipes, different equipment, different measuring conventions. The question is: can reading the public book make us better at predicting outcomes in our kitchen? The answer turns out to be yes, but only if you are careful about which parts of the public book actually apply to your kitchen.


## Chapter 2: Setup

**Data split by year** — the `year` column encodes which dataset a row belongs to: `year == 2025` = our experiments; `year <= 2019` = published literature.

---

### Why not Ridge or RandomForest?

- **Ridge (linear regression)** assumes Y(C2) is a weighted sum of element percentages. It cannot detect that Ba + La *together* behave differently from each alone — catalyst chemistry is full of such non-linear interactions. Expected RMSE would be >4%.
- **RandomForestRegressor** builds decision trees independently on bootstrap samples — each tree does not know what the others got wrong. Gradient boosting (XGBoost, LightGBM) builds trees *sequentially*, each one correcting the residuals of all previous ones. On this dataset, gradient boosting gives ~30–40% lower RMSE than Random Forest.

---

### Three critical setup decisions

**1. `LabelEncoder`** — The `Preparation` column is text ("Impregnation", "Sol-gel", etc.). Tree models need integers. We fit the encoder on the full combined vocabulary, then apply it to each subset.

**2. `StandardScaler`** — Element weights span 0–15%; temperature spans 500–900°C. Without scaling, temperature would dominate distance calculations simply because its units are larger. After scaling, one unit = one standard deviation, regardless of the original feature's range.

**3. Fit on ours, apply to both** — `scaler.fit_transform(X_ours)` establishes our data as the reference ruler. We then call `scaler.transform(X_lit)` — the *same* ruler applied to literature. If we re-fit the scaler on literature separately, the identical Ba=2% measurement would get a different scaled value depending on which dataset it came from. That would break every distance-based calculation downstream.

---

> **Intuition:** The StandardScaler step is easy to miss but important. Imagine measuring temperature in Celsius and element weight in grams — if temperature ranges 500–900 and weight ranges 0–15, distance calculations treat temperature as 50× more important just because of units. Scaling puts everything on the same footing: a one-unit change in any feature represents one standard deviation in that feature. Both datasets must use the same scale — the one learned from our data — otherwise a Ba=2% sample from literature would get a different scaled value than the identical Ba=2% sample from our lab.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
import shap
from scipy.optimize import minimize
from scipy.spatial.distance import cdist
from scipy.stats import rankdata, gaussian_kde
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)
plt.rcParams['figure.dpi'] = 110

DATA_PATH = 'OCM_lab_data_and_literature_datal.csv'
df = pd.read_csv(DATA_PATH)

# Split by year — the year column IS the dataset label
df_ours = df[df['year'] == 2025].copy().reset_index(drop=True)
df_lit  = df[df['year'] <= 2019].copy().reset_index(drop=True)

print(f"Our data   : {len(df_ours):,} rows  (mean Y={df_ours['Y(C2), %'].mean():.2f}%)")
print(f"Literature : {len(df_lit):,} rows  (mean Y={df_lit['Y(C2), %'].mean():.2f}%)")
print(f"Label shift: {df_lit['Y(C2), %'].mean() - df_ours['Y(C2), %'].mean():.2f} pp")

TARGET    = 'Y(C2), %'
ELEM_COLS = [c for c in df.columns if c not in ['Preparation', 'Temperature_C', TARGET, 'year']]

# LabelEncoder: trees need integers, not strings, for Preparation method
le = LabelEncoder()
le.fit(df['Preparation'])                             # fit on full vocabulary
df_ours['prep_enc'] = le.transform(df_ours['Preparation'])
df_lit['prep_enc']  = le.transform(df_lit['Preparation'])

FEATURES  = ['Temperature_C', 'prep_enc'] + ELEM_COLS
X_ours    = df_ours[FEATURES].values.astype(float)
y_ours    = df_ours[TARGET].values
X_lit     = df_lit[FEATURES].values.astype(float)
y_lit     = df_lit[TARGET].values

# fit_transform on OURS, only transform on literature — same reference ruler
scaler    = StandardScaler()
X_ours_sc = scaler.fit_transform(X_ours)
X_lit_sc  = scaler.transform(X_lit)


### Visual evidence of the two shifts

The KDE on the left shows the **label shift**: both curves are normalised so the difference in sample size doesn't dominate, and the red band marks the 3.42 pp mean gap as a physical distance. The PCA on the right shows the **covariate shift**: a 2D projection of the 67-dimensional feature space, with our data (blue) clustered tightly and literature (orange) spread across regions our data never visits.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

# ── Panel 1: Label shift — Y(C2) KDE ────────────────────────────────────────
ax = axes[0]
BLUE, ORANGE = '#2196F3', '#FF9800'
mean_ours, mean_lit = y_ours.mean(), y_lit.mean()

ax.hist(y_ours, bins=60, density=True, alpha=0.35, color=BLUE,   label='Our data')
ax.hist(y_lit,  bins=40, density=True, alpha=0.45, color=ORANGE, label='Literature')
for y, color in [(y_ours, BLUE), (y_lit, ORANGE)]:
    kde = gaussian_kde(y, bw_method=0.15)
    xs  = np.linspace(0, y.max(), 400)
    ax.plot(xs, kde(xs), color=color, lw=2.5)

ax.axvline(mean_ours, color=BLUE,   ls='--', lw=2, label=f'Mean ours = {mean_ours:.2f}%')
ax.axvline(mean_lit,  color=ORANGE, ls='--', lw=2, label=f'Mean lit  = {mean_lit:.2f}%')
ax.axvspan(mean_ours, mean_lit, alpha=0.12, color='red',
           label=f'Shift = {mean_lit-mean_ours:.2f} pp')
ax.set_xlim(0, 30)
ax.set_xlabel('Y(C2) [%]'); ax.set_ylabel('Probability density')
ax.set_title('Label shift — Y(C2) distribution')
ax.legend(fontsize=9)

# ── Panel 2: Covariate shift — PCA of feature space ─────────────────────────
ax = axes[1]
idx_sub       = np.random.choice(len(X_ours_sc), size=3000, replace=False)
X_combined_sc = np.vstack([X_ours_sc[idx_sub], X_lit_sc])
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_combined_sc)

ax.scatter(X_pca[:3000, 0], X_pca[:3000, 1],
           alpha=0.15, s=8, color='steelblue', label='Our data (n=3000)')
ax.scatter(X_pca[3000:, 0], X_pca[3000:, 1],
           alpha=0.4,  s=12, color='darkorange', label=f'Literature (n={len(X_lit_sc)})')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('Covariate shift — PCA of catalyst feature space')
ax.legend(markerscale=2)

plt.tight_layout()
plt.show()


### What chemistry differs?

The PCA above shows *that* the chemistry differs, but not *which* elements drive the gap. Here we plot the top 15 elements by usage frequency (fraction of samples with non-zero loading) in each dataset side-by-side. The asymmetry is concrete: our lab leans on one set of promoters and active phases; literature has a different and broader palette. This is why filtering by chemical similarity (Chapter 5) is necessary — large parts of the literature dataset describe chemistry our lab has never tested.

In [ ]:
# Top elements by usage frequency in each dataset
nonzero_ours = (df_ours[ELEM_COLS] > 0).mean().sort_values(ascending=False)
nonzero_lit  = (df_lit[ELEM_COLS] > 0).mean().sort_values(ascending=False)

top_n = 15
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

top_ours = nonzero_ours.head(top_n)
axes[0].barh(top_ours.index[::-1], top_ours.values[::-1],
             color='steelblue', edgecolor='k')
axes[0].set_xlabel('Fraction of samples with element > 0')
axes[0].set_title(f'Top {top_n} elements — Our data')

top_lit = nonzero_lit.head(top_n)
axes[1].barh(top_lit.index[::-1], top_lit.values[::-1],
             color='darkorange', edgecolor='k')
axes[1].set_xlabel('Fraction of samples with element > 0')
axes[1].set_title(f'Top {top_n} elements — Literature')

plt.tight_layout()
plt.show()


## Chapter 3: How We Measure Success

**Metric:** RMSE (root-mean-squared error) in units of **percentage points of C₂ yield**. An RMSE of 2.1 means the model is typically off by about 2 percentage points.

---

### Asymmetric 5-fold cross-validation

Our 89,074 samples are split into 5 equal folds. In each of the 5 rounds:

- **Training fold**: 4/5 of our data + any literature samples being tested
- **Validation fold**: 1/5 of our **data only** — literature never appears here

This asymmetry is deliberate. The question we are answering is:

> "How accurately can this model predict *our* experiments?"

If literature samples appeared in validation, we would be measuring accuracy on a mix of our data and literature — diluting and distorting the answer. The validation set must reflect the actual deployment target.

The code below defines `evaluate_cv_ours`, which is called identically for every method. The only thing that changes between methods is what gets passed as `X_train_extra` and `y_train_extra`.

---

> **Intuition:** Cross-validation is like a student doing practice exams with different question sets. Each time, 4/5 of the questions are for studying and 1/5 are for testing — and you rotate which fifth is the test. The asymmetric part here is that literature is always in the "study" pile — it never appears in the test pile. This is deliberate: we are measuring how well the model predicts our experiments, not some mix of ours and literature. If literature appeared in validation, we would be measuring a diluted version of the wrong question.


In [ ]:
def lgb_params():
    return dict(
        n_estimators=500, learning_rate=0.05, num_leaves=63,
        max_depth=7, min_child_samples=20, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, verbosity=-1
    )


def evaluate_cv_ours(X_train_extra=None, y_train_extra=None,
                     sample_weight_extra=None, label='', cv=5):
    kf = KFold(n_splits=cv, shuffle=True, random_state=42)
    rmse_list, r2_list = [], []

    for train_idx, val_idx in kf.split(X_ours_sc):
        X_val_f = X_ours_sc[val_idx]      # validation: our data only, always
        y_val_f = y_ours[val_idx]

        if X_train_extra is not None:
            X_tr_f = np.vstack([X_ours_sc[train_idx], X_train_extra])
            y_tr_f = np.concatenate([y_ours[train_idx], y_train_extra])
            sw_ours = np.ones(len(train_idx))
            sw = (np.concatenate([sw_ours, sample_weight_extra])
                  if sample_weight_extra is not None else None)
        else:
            X_tr_f = X_ours_sc[train_idx]
            y_tr_f = y_ours[train_idx]
            sw     = None

        model = lgb.LGBMRegressor(**lgb_params())
        model.fit(X_tr_f, y_tr_f, sample_weight=sw)
        preds = model.predict(X_val_f)

        rmse_list.append(np.sqrt(mean_squared_error(y_val_f, preds)))
        r2_list.append(r2_score(y_val_f, preds))

    rmse_m = np.mean(rmse_list)
    r2_m   = np.mean(r2_list)
    rmse_s = np.std(rmse_list)
    print(f"  {label:50s}  RMSE={rmse_m:.4f} \u00b1 {rmse_s:.4f}   R\u00b2={r2_m:.4f}")
    return rmse_m, r2_m, rmse_s


results = {}


## Chapter 4: Baseline + Why Naive Merging Fails

### Step 1 — Establish the baseline

First we measure how well LightGBM predicts our experiments using **only our own data**. This gives us the number every subsequent method must beat.

| Method | CV RMSE | Δ vs baseline |
|---|---|---|
| Baseline (our data only) | **2.133** | — |

This is our reference point. If a method produces RMSE > 2.133, it has *harmed* the model.


### Step 2 — The obvious attempt: add all literature

Before building anything complicated, the natural first step is: **append all 3,852 literature rows to the training set.** More data should help. It didn't.

| Method | CV RMSE | Δ vs baseline |
|---|---|---|
| Baseline | 2.133 | — |
| Naive merge (all 3,852 lit rows) | **2.248** | +0.115 (**5% worse**) |

---

**Why it failed:**

The model now sees two catalysts with nearly identical element compositions:
- One labelled **5.2%** (from our lab)
- One labelled **8.7%** (from literature)

There is no feature in the dataset that explains the 3.42 pp gap — it comes from unmeasured operating conditions (gas flow rate, CH₄:O₂ ratio) that were set differently across labs. The model has no way to distinguish which label belongs to which context, so it hedges and predicts both poorly.

The 3.42 pp offset is not noise — it is a **systematic signal** that the model cannot account for and incorrectly tries to fit. This is exactly what the KDE plot in Chapter 2 shows: the two distributions are not the same.

**Conclusion:** We cannot simply mix the data. We need to either (a) be selective about *which* literature to add, or (b) prevent the offset from contaminating the training labels entirely.

The next three methods each try a different answer.

---

> **Intuition:** Imagine teaching a student by mixing two textbooks — one from your university's curriculum and one from a university with completely different grading standards and exam formats. The student now gets contradictory signals: the same concept is explained differently, scored differently, and the exam questions don't match either book cleanly. They end up worse than if they had just used the first textbook. That is exactly what happens here — the model sees similar catalyst compositions labelled ~5% in our data and ~10% in literature, with no feature to explain the discrepancy, and hedges badly.


In [ ]:
rmse, r2, std = evaluate_cv_ours(label='Baseline (our data only)')
results['1. Baseline (ours only)'] = (rmse, r2, std)
baseline_rmse = rmse

rmse, r2, std = evaluate_cv_ours(
    X_train_extra=X_lit_sc,
    y_train_extra=y_lit,
    label='Naive merge (all literature)')
results['2. Naive merge (all lit)'] = (rmse, r2, std)

print(f"\nBaseline RMSE    : {baseline_rmse:.4f}")
print(f"Naive merge RMSE : {results['2. Naive merge (all lit)'][0]:.4f}  "
      f"(\u0394 = {results['2. Naive merge (all lit)'][0] - baseline_rmse:+.4f})")


## Chapter 5: DRST — Filtering by Chemical Similarity

**Idea:** Instead of adding all 3,852 literature samples, only add the ones that *look chemically like our data*. Score each literature sample: "how much does this look like one of our catalysts?"

---

### How DRST (Density-Ratio Selective Transfer) works

1. Build a binary classifier: given a feature vector, can it tell whether the sample came from our lab (`label=1`) or from literature (`label=0`)?
2. Score every literature sample: `P(ours | x)` — the probability the classifier assigns the label "ours" to this sample.
3. Keep only literature samples that score above a threshold τ.

```python
clf_dom = LogisticRegression(C=0.5)
clf_dom.fit(X_dom, y_dom)                            # ours=1, literature=0
p_ours_lit = clf_dom.predict_proba(X_lit_sc)[:, 1]  # P(ours | this sample)
mask = p_ours_lit >= 0.30                            # 782 of 3852 samples pass
```

**Why `C=0.5`?** C controls regularisation — smaller C = smoother decision boundary. We want a *general* boundary capturing the overall chemical difference between datasets, not one that memorises individual samples.

**Why `predict_proba` not `predict`?** We want a continuous score (0.00–1.00), not a binary decision. The score itself is meaningful.

**Why τ = 0.30?** We swept five candidate thresholds {0.05, 0.10, 0.20, 0.30, 0.40} and selected the one that minimised CV RMSE. τ = 0.30 retains 782 of 3,852 samples (~20%).

The histogram below shows where literature samples land on the P(ours|x) axis. The four dashed lines mark candidate thresholds — you can see most samples cluster at low probabilities (the chemistry the classifier confidently calls "literature"), with a tail extending toward our distribution.

---

| Method | CV RMSE | Δ vs baseline |
|---|---|---|
| Baseline | 2.133 | — |
| Naive merge | 2.248 | +5.4% |
| **DRST (τ=0.30)** | **2.019** | **−5.3%** |

**What it leaves unsolved:** The cutoff is hard — a sample scoring 0.29 is completely discarded while a sample scoring 0.31 gets full weight. This binary decision feels arbitrary. Can we do something softer?

---

> **Intuition:** Think of DRST like a bouncer at a door. The club inside is our lab's chemical space. The bouncer doesn't know each person personally — instead it has learned what the regulars look like (element profiles of our data) and makes a judgment call for each newcomer. A literature sample with Ba, La, and similar temperature range walks in and scores high — it looks familiar. A literature sample dominated by Si and Na at extreme temperatures scores low — it looks like it belongs somewhere else. We let in only those scoring above 0.30.


In [ ]:
# Build a domain classifier: ours=1, literature=0
rng     = np.random.default_rng(42)
sub_idx = rng.choice(len(X_ours_sc), size=min(10_000, len(X_ours_sc)), replace=False)

X_dom = np.vstack([X_ours_sc[sub_idx], X_lit_sc])
y_dom = np.concatenate([np.ones(len(sub_idx)), np.zeros(len(X_lit_sc))])

clf_dom    = LogisticRegression(C=0.5, max_iter=1000, random_state=42, n_jobs=-1)
clf_dom.fit(X_dom, y_dom)

p_ours_lit = clf_dom.predict_proba(X_lit_sc)[:, 1]   # P(ours | this sample)

# Visualise the score distribution and candidate thresholds
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(p_ours_lit, bins=40, color='teal', edgecolor='k', alpha=0.8)
for tau in [0.10, 0.20, 0.30, 0.40]:
    ax.axvline(tau, ls='--', label=f'\u03c4={tau}')
ax.set_xlabel('P(our data | x)  — density-ratio score')
ax.set_ylabel('Number of literature samples')
ax.set_title('DRST: how many literature samples look like our chemistry?')
ax.legend()
plt.tight_layout()
plt.show()

print(f"P(ours|x):  mean={p_ours_lit.mean():.3f}  median={np.median(p_ours_lit):.3f}")
for tau in [0.10, 0.20, 0.30, 0.40, 0.50]:
    n = (p_ours_lit >= tau).sum()
    print(f"  \u03c4={tau:.2f}: {n:4d} / {len(p_ours_lit)} samples retained ({100*n/len(p_ours_lit):.1f}%)")


In [ ]:
# Sweep tau — select by lowest CV RMSE
best_drst = (np.inf, 0, 0, None)

for tau in [0.05, 0.10, 0.20, 0.30, 0.40]:
    mask = p_ours_lit >= tau
    if mask.sum() == 0:
        continue
    rmse, r2, std = evaluate_cv_ours(
        X_train_extra=X_lit_sc[mask],
        y_train_extra=y_lit[mask],
        label=f'DRST \u03c4={tau:.2f} ({mask.sum()} lit samples)')
    if rmse < best_drst[0]:
        best_drst = (rmse, r2, std, tau)

results[f'3. DRST (\u03c4={best_drst[3]})'] = (best_drst[0], best_drst[1], best_drst[2])
print(f"\n\u2192 Best \u03c4 = {best_drst[3]}")
print(f"\u2192 DRST RMSE = {best_drst[0]:.4f}  (\u0394 vs baseline = {best_drst[0]-baseline_rmse:+.4f})")


## Chapter 6: KMM — Soft Weights Instead of a Hard Filter

**Idea:** Instead of keep/discard, assign each literature sample a weight between 0 and 10. Samples close to our distribution → weight near 1. Samples far away → weight near 0. The model trains on all literature but gives very little influence to foreign samples.

---

### The mathematical core

KMM uses an RBF (Radial Basis Function) kernel, which measures similarity by distance — two samples with similar element profiles get a high similarity score:

```python
K_ss = rbf_kernel(X_lit, X_lit, σ)     # how similar are lit samples to each other?
K_st = rbf_kernel(X_lit, X_ours, σ)    # how similar is each lit sample to our data?
κ    = (n_lit/n_ours) * K_st.sum(axis=1)  # κᵢ = how much does lit_i overlap with our distribution?
```

We then minimise: **½ wᵀ K\_ss w − κ · w** subject to 0 ≤ wᵢ ≤ 10.

The objective pulls weights toward samples with high κ (overlap with our data) and lets weights for foreign samples converge to zero.

**σ (bandwidth)** is set by the *median heuristic*: σ = median of all pairwise distances in a subsample of both datasets combined. This gives a scale that reflects the actual spread of the data.

---

| Method | CV RMSE | Δ vs baseline |
|---|---|---|
| Baseline | 2.133 | — |
| DRST (τ=0.30) | 2.019 | −5.3% |
| **KMM** | **2.035** | **−4.6%** |

**Key finding:** Both DRST and KMM independently flag the same ~78.5% of literature samples as near-useless. The scatter plot below shows the KMM weight against the DRST score for every literature sample — points cluster along a clear positive relationship (**r = 0.79**). Two completely different mathematical approaches — one a logistic classifier, one a quadratic optimisation — reaching the same conclusion gives strong confidence this is a real signal, not an artifact.

**What it leaves unsolved:** Both DRST and KMM still add literature *labels* directly to the training set. Those labels carry the 3.42 pp systematic offset. Filtering reduces the damage but does not eliminate the root cause.

---

> **Intuition:** Where DRST gives each sample a yes/no vote, KMM gives each sample a vote weight between 0 and 10. The weight is determined by solving an optimisation problem: find weights such that the weighted average of literature samples, in a kernel similarity space, matches the average of our data as closely as possible. Samples that land near our data cluster get pulled toward high weights. Samples far away have nothing to match in our data, so their weight converges to near zero. The interesting result is that both DRST and KMM end up agreeing on which samples are useful — two completely different mathematical approaches, same conclusion.


In [ ]:
def rbf_kernel(X, Y, sigma):
    dist_sq = cdist(X, Y, metric='sqeuclidean')
    return np.exp(-dist_sq / (2 * sigma ** 2))


def kmm_weights(X_source, X_target, B=5.0, n_target_sub=5000):
    n_s   = len(X_source)
    rng2  = np.random.default_rng(42)
    t_idx = rng2.choice(len(X_target), size=min(n_target_sub, len(X_target)), replace=False)
    X_tgt = X_target[t_idx];  n_t = len(X_tgt)

    # sigma via median heuristic
    idx_s = rng2.choice(n_s, size=min(2000, n_s), replace=False)
    X_sub = np.vstack([X_source[idx_s], X_tgt[:1000]])
    d     = cdist(X_sub, X_sub, metric='euclidean')
    sigma = float(np.median(d[d > 0]))
    print(f"  \u03c3 = {sigma:.3f}")

    K_ss  = rbf_kernel(X_source, X_source, sigma)   # lit-to-lit similarity
    K_st  = rbf_kernel(X_source, X_tgt,    sigma)   # lit-to-ours similarity
    kappa = (n_s / n_t) * K_st.sum(axis=1)          # overlap with our distribution

    eps   = (np.sqrt(n_s) - 1) / np.sqrt(n_s)

    def obj(w):
        return (0.5 * w @ K_ss @ w - kappa @ w
                + 1e4 * max(0.0, abs(w.sum() / n_s - 1.0) - eps) ** 2)

    def grad(w):
        g    = K_ss @ w - kappa
        diff = w.sum() / n_s - 1.0
        if abs(diff) > eps:
            g += 1e4 * 2.0 * diff / n_s * np.ones(n_s)
        return g

    res = minimize(obj, np.ones(n_s), jac=grad, method='L-BFGS-B',
                   bounds=[(0.0, B)] * n_s,
                   options={'maxiter': 1000, 'ftol': 1e-9, 'gtol': 1e-6})
    return np.clip(res.x, 0.0, B)


print("Computing KMM weights (this takes ~1-2 minutes)...")
w_kmm  = kmm_weights(X_lit_sc, X_ours_sc)
r_corr = np.corrcoef(p_ours_lit, w_kmm)[0, 1]
print(f"Near-zero weights (w < 0.1): {(w_kmm < 0.1).sum()} / {len(w_kmm)} "
      f"({100*(w_kmm < 0.1).mean():.1f}%)")
print(f"Correlation with DRST scores: r = {r_corr:.3f}")

# Visualise: do KMM and DRST agree?
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
axes[0].hist(w_kmm, bins=40, color='mediumpurple', edgecolor='k', alpha=0.8)
axes[0].set_xlabel('KMM weight')
axes[0].set_ylabel('Number of literature samples')
axes[0].set_title('Distribution of KMM weights')

axes[1].scatter(p_ours_lit, w_kmm, alpha=0.3, s=8, color='purple')
axes[1].set_xlabel('DRST score  P(ours|x)')
axes[1].set_ylabel('KMM weight')
axes[1].set_title(f'KMM vs DRST agreement   (r = {r_corr:.3f})')
plt.tight_layout()
plt.show()


In [ ]:
rmse, r2, std = evaluate_cv_ours(
    X_train_extra=X_lit_sc,
    y_train_extra=y_lit,
    sample_weight_extra=w_kmm,
    label='KMM weighted literature')
results['4. KMM weighted'] = (rmse, r2, std)
print(f"\n\u2192 KMM RMSE = {rmse:.4f}  (\u0394 vs baseline = {rmse-baseline_rmse:+.4f})")


## Chapter 7: Prior Feature Transfer — The Winner

**Why DRST and KMM both hit a ceiling:** Both methods filter or down-weight foreign literature, but they still pass literature *labels* directly into the training objective. The 3.42 pp systematic offset bleeds through into the model's parameters regardless.

**The key insight:** What if we don't expose literature labels to the final model at all?

---

### Two-stage design

**Stage 1 — Literature expert (XGBoost, trained on DRST-filtered literature only):**

Train a model on the 782 literature samples that DRST identified as chemically relevant. This model learns the *literature's view* of catalyst chemistry: given a composition, what yield does published literature say is typical?

*XGBoost is used for Stage 1 because it is more conservative on small data (782 rows) than LightGBM.*

**Stage 2 — Our model, with the expert's opinion as a new feature (LightGBM, trained on our labels only):**

For every catalyst in our training set, ask Stage 1: "what would you predict for this composition?" That prediction becomes a **68th feature** called `lit_prior_prediction`. Stage 2 then trains on *our labels* only, with this extra feature available.

Stage 2 can learn to:
- Trust the prior in regions where literature and our experiments agree
- Discount the prior in regions where the offset is large
- Ignore the prior for chemistries unlike anything in literature

**Why this is better than routing literature labels directly:**
The 3.42 pp offset affects the prior feature's *value*, not our training *labels*. Stage 2 can learn the offset as a calibration problem rather than having it corrupt the loss function.

We also quantile-normalize literature labels to our distribution before Stage 1 training — this aligns the label scale without destroying the rank ordering of high vs. low yield catalysts.

---

| Method | CV RMSE | Δ vs baseline | OOD RMSE |
|---|---|---|---|
| Baseline | 2.133 | — | 6.53 |
| DRST | 2.019 | −5.3% | — |
| KMM | 2.035 | −4.6% | — |
| **Prior Feature Transfer** | **1.907** | **−10.6%** | **3.60 (−45%)** |

**Why is OOD improvement so large (−45%)?** Stage 1 was trained on literature covering sol-gel, coprecipitation, and other preparation methods. When Stage 2 encounters an unfamiliar composition, the prior feature provides a meaningful starting estimate from the literature domain. Without the prior, Stage 2 would be extrapolating blind.

---

> **Intuition:** Think of it like a student consulting an expert before an exam. The expert (Stage 1) has read all the published literature and forms an opinion: "given your element combination, I'd expect around X% yield based on what I've seen." That opinion becomes one piece of evidence for the student (Stage 2), who has also done 89,000 of their own experiments. Stage 2 can choose to trust or discount the expert's opinion based on whether it aligns with what they've seen themselves. Crucially, the expert's opinion influences the prediction — it does not replace the student's own experience. The student's grades are measured against the student's own results, not the expert's history.


In [ ]:
def quantile_normalize_y(y_source, y_target):
    """Map y_source onto y_target's distribution via rank-matching."""
    quantiles = rankdata(y_source, method='average') / (len(y_source) + 1)
    return np.quantile(y_target, np.clip(quantiles, 0.01, 0.99))


def xgb_params():
    return dict(n_estimators=400, learning_rate=0.05, max_depth=6,
                subsample=0.8, colsample_bytree=0.8,
                reg_alpha=0.1, reg_lambda=1.0,
                random_state=42, verbosity=0, n_jobs=-1)


# Use DRST-filtered literature for Stage 1 (tau = best_drst[3])
mask = p_ours_lit >= best_drst[3]
print(f"Stage-1 literature: {mask.sum()} samples (tau={best_drst[3]})")

rmse_list, r2_list = [], []

for train_idx, val_idx in KFold(5, shuffle=True, random_state=42).split(X_ours_sc):
    # Quantile-normalise literature labels to our fold's distribution
    y_lit_qn = quantile_normalize_y(y_lit[mask], y_ours[train_idx])

    # ── Stage 1: XGBoost learns literature's view of chemistry ──────────────
    pre = xgb.XGBRegressor(**xgb_params())
    pre.fit(X_lit_sc[mask], y_lit_qn)          # literature only, DRST-filtered

    # Generate the "literature opinion" for each of our training/validation samples
    prior_tr  = pre.predict(X_ours_sc[train_idx]).reshape(-1, 1)
    prior_val = pre.predict(X_ours_sc[val_idx]).reshape(-1, 1)

    # ── Stage 2: LightGBM on our labels, with prior as 68th feature ─────────
    final = lgb.LGBMRegressor(**lgb_params())
    final.fit(
        np.hstack([X_ours_sc[train_idx], prior_tr]),   # 67 features + prior
        y_ours[train_idx])                              # OUR labels — not literature's
    preds = final.predict(np.hstack([X_ours_sc[val_idx], prior_val]))

    rmse_list.append(np.sqrt(mean_squared_error(y_ours[val_idx], preds)))
    r2_list.append(r2_score(y_ours[val_idx], preds))

rmse_ft = np.mean(rmse_list)
r2_ft   = np.mean(r2_list)
std_ft  = np.std(rmse_list)
results['5. Prior Feature Transfer'] = (rmse_ft, r2_ft, std_ft)

print(f"\nPrior Feature Transfer  RMSE = {rmse_ft:.4f} \u00b1 {std_ft:.4f}   R\u00b2 = {r2_ft:.4f}")
print(f"\u0394 vs baseline = {rmse_ft - baseline_rmse:+.4f}  "
      f"({100*(rmse_ft-baseline_rmse)/baseline_rmse:+.1f}%)")


## Chapter 8: Results + SHAP — Did It Learn Real Chemistry?

### Results summary

| Method | CV RMSE | Δ vs baseline |
|---|---|---|
| Baseline (ours only) | 2.133 | — |
| Naive merge | 2.248 | +5.4% |
| DRST (τ=0.30) | 2.019 | −5.3% |
| KMM | 2.035 | −4.6% |
| **Prior Feature Transfer** | **1.907** | **−10.6%** |

The horizontal bar chart below makes the trajectory visible at a glance: naive merging pushes us above baseline (red region); selective transfer drops us below it; Prior Feature Transfer is the largest improvement.

---

### SHAP: does the model use real chemistry?

RMSE tells you *how wrong* the model is on average. SHAP tells you *why* it makes each individual prediction.

For each prediction, SHAP (SHapley Additive exPlanations) asks: if we imagine all possible orderings of features being revealed one at a time, what is the average marginal contribution of each feature to this prediction? A positive SHAP value means "this feature, at this value, pushed the prediction higher than the baseline." Negative means it pushed lower.

The beeswarm plot shows all 3,000 samples at once:
- Each **row** is a feature (ranked by mean absolute SHAP value)
- Each **dot** is one sample
- **Horizontal position** = effect on prediction (right = pushes yield up)
- **Colour** = feature value (red = high, blue = low)

**Key findings:**

1. `lit_prior_prediction` is the **#1 most impactful feature** — the transfer learning is genuinely being used, not sitting idle as dead weight.
2. **Temperature** is second — higher temperature pushes yield up, consistent with OCM thermodynamics.
3. **Ba, Mn, La, Ce** all appear as important positive contributors — these are known OCM active phases and promoters.

**One concern — ceiling effect at Y > 15%:**
The model systematically under-predicts by ~4 pp for high-yield catalysts. This is not a modelling failure — it is a *data limitation*. The conditions that explain extreme yields (gas flow rate GHSV, CH₄:O₂ ratio) are not in the feature set. No algorithm can compensate for a missing column.

---

> **Intuition:** RMSE tells you how wrong the model is on average. SHAP tells you *why*. For each prediction, SHAP asks: if we imagine all possible orderings of features being revealed one at a time, what is the average marginal contribution of this particular feature to this particular prediction? A positive SHAP value means this feature pushed the prediction higher than average; negative means it pushed lower. The beeswarm shows all 3,000 samples at once — one dot per sample per feature — so you can see not just the average effect but the full spread and direction. If the model has learned real chemistry, the physics should show up here.


In [ ]:
# ── Results bar chart: visual comparison of all 5 methods ──────────────────
df_res = pd.DataFrame(
    [(k, v[0], v[1], v[2]) for k, v in results.items()],
    columns=['Method', 'CV RMSE', 'CV R\u00b2', 'RMSE std']).sort_values('CV RMSE')

palette = ['#2196F3' if 'Baseline' in m else
           '#D32F2F' if 'Naive' in m else
           '#FF9800' for m in df_res['Method']]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(df_res['Method'], df_res['CV RMSE'],
        xerr=df_res['RMSE std'], color=palette,
        edgecolor='k', capsize=3)
ax.axvline(baseline_rmse, color='steelblue', ls='--', alpha=0.7, label='Baseline RMSE')
ax.set_xlabel('CV RMSE [%]  (lower = better)')
ax.set_title('All 5 methods — 5-fold CV on our data')
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── SHAP analysis: does the model use real chemistry? ──────────────────────
y_lit_qn_all = quantile_normalize_y(y_lit[mask], y_ours)

pre_shap = xgb.XGBRegressor(**xgb_params())
pre_shap.fit(X_lit_sc[mask], y_lit_qn_all)     # literature only, DRST-filtered

prior_all = pre_shap.predict(X_ours_sc).reshape(-1, 1)
X_aug     = np.hstack([X_ours_sc, prior_all])
aug_feat_names = FEATURES + ['lit_prior_prediction']

final_shap = lgb.LGBMRegressor(**lgb_params())
final_shap.fit(X_aug, y_ours)

print("Computing SHAP values...")
rng_sh = np.random.default_rng(42)
idx_sh = rng_sh.choice(len(X_aug), size=3000, replace=False)
X_sh   = X_aug[idx_sh]

explainer   = shap.TreeExplainer(final_shap)   # exact, not approximate
shap_values = explainer.shap_values(X_sh)      # 3000-sample subsample

shap.summary_plot(shap_values, X_sh, feature_names=aug_feat_names,
                  max_display=15, show=False)
plt.title("SHAP Beeswarm — Prior Feature Transfer model")
plt.tight_layout()
plt.show()
print(f"\nSHAP matrix shape: {shap_values.shape}  ({len(aug_feat_names)} features)")
